In [2]:
from unidecode import unidecode
import random

In [3]:
dictionary = open('oxford_dictionary.txt', 'r', encoding="utf-8").read().splitlines()
dictionary[:5]

['A ',
 '',
 '',
 '',
 'A-  prefix (also an- before a vowel sound) not, without (amoral). [greek]']

In [4]:
to_remove = [
    '[', ']', '(', ')',
    '-sing', '-ting', '-cuses', 
    '—v.', ' v.' , 
    '—n.', ' n.',
    '—adv.', ' adv.', 
    '—adj.', ' adj.', 
    ' pl.', '-s', '-ies',
    ' abbr.', '—prep', '—ad', ' naut.', '-baft',
    ': related to', '\x7f',
]

def cleanStr(s):
    if 'prefix' in s or s == '':
        return

    l = s.find('[')
    r = s.find(']')
    if l < r and r > -1:
        s = s[:l] + s[r+1:]
        l = s.find('[')
        r = s.find(']')
    
    l = s.find('(')
    r = s.find(')')
    if l < r and r > -1:
        s = s[:l] + s[r+1:]
        l = s.find('(')
        r = s.find(')')

    s = s.split(' ')        
    s = [w for w in s if '*' not in w]
    s = ' '.join(s)
    
    for r in to_remove:
        s = s.replace(r,"") 
    while '  ' in s:
        s = s.replace('  ', ' ')    
    while '. .' in s:
        s = s.replace('. .', '.') 
    while ' .' in s:
        s = s.replace(' .', '.')    
    
    if s[-1] in (',',':'):
        s = s[:len(s)-1]

    s = unidecode(s)
    s = s.strip()
    
    if len(s) == 1:
        return
    return s

dict_clean = [cleanStr(ele) for ele in dictionary]
dict_clean = list(filter(None, dict_clean))
data = dict_clean
random.seed(770)
random.shuffle(data)
data[:5]

['Cox Coxswain, esp. Of a racing-boat. Act as cox.',
 'Investigate 1 inquire into; examine. 2 make a systematic inquiry. investigation Investigative Investigator Investigatory',
 'Right wing 1 more conservative section of a political party or system. 2 right side of a football etc. Team on the field. conservative or reactionary. right-winger',
 'Gatt General agreement on tariffs and trade.',
 'Witticism Witty remark.']

In [5]:
chars = sorted(list(set(''.join(data))))
stoi = {s:i+2 for i,s in enumerate(chars)}
stoi['<s>'] = 0
stoi['<e>'] = 1
stoi = dict(sorted(stoi.items(), key = lambda x: x[1]))
itos = {i:s for s,i in stoi.items()}
print(itos)
print('vocab:', len(itos))

{0: '<s>', 1: '<e>', 2: ' ', 3: '!', 4: '"', 5: '$', 6: '%', 7: '&', 8: "'", 9: '(', 10: ')', 11: '+', 12: ',', 13: '-', 14: '.', 15: '/', 16: '0', 17: '1', 18: '2', 19: '3', 20: '4', 21: '5', 22: '6', 23: '7', 24: '8', 25: '9', 26: ':', 27: ';', 28: '=', 29: '?', 30: 'A', 31: 'B', 32: 'C', 33: 'D', 34: 'E', 35: 'F', 36: 'G', 37: 'H', 38: 'I', 39: 'J', 40: 'K', 41: 'L', 42: 'M', 43: 'N', 44: 'O', 45: 'P', 46: 'Q', 47: 'R', 48: 'S', 49: 'T', 50: 'U', 51: 'V', 52: 'W', 53: 'X', 54: 'Y', 55: 'Z', 56: '^', 57: 'a', 58: 'b', 59: 'c', 60: 'd', 61: 'e', 62: 'f', 63: 'g', 64: 'h', 65: 'i', 66: 'j', 67: 'k', 68: 'l', 69: 'm', 70: 'n', 71: 'o', 72: 'p', 73: 'q', 74: 'r', 75: 's', 76: 't', 77: 'u', 78: 'v', 79: 'w', 80: 'x', 81: 'y', 82: 'z', 83: '{', 84: '|', 85: '}'}
vocab: 86


In [6]:
class Tokenizer:
    def __init__(self, data):
        chars = sorted(list(set(''.join(data))))
        encoding = {s:i+2 for i,s in enumerate(chars)}
        encoding['<s>'] = 0
        encoding['<e>'] = 1
        self.encoding = dict(sorted(encoding.items(), key = lambda x: x[1]))
        self.decoding = {i:s for s,i in self.encoding.items()}
        self.bpe_encoding = {}
        self.bpe_decoding = {}                

    def encode(self, data):
        return [list(self.encoding[s] for s in d) for d in data]
    
    def decode(self, encoded):
        encoded = self.bytePairDecode(encoded)
        return [''.join(self.decoding[i] for i in d) for d in encoded]

    def bytePairEncode(self, encoded):
        freq = {}
        for ele in encoded:
            n = len(ele)
            for i in range(n-1):
                pair = (ele[i],ele[i+1])
                freq[pair] = freq.get(pair,0) + 1
        max_pair = (None,0)
        for k,v in freq.items():
            if v > max_pair[1]:
                max_pair = (k,v)
        new_int = len(self.encoding) + len(self.bpe_encoding)
        self.bpe_encoding[max_pair[0]] = new_int
        self.bpe_decoding[new_int] = max_pair[0]
        
        def updateEncoding(ele):
            res = []
            n = len(ele)
            i = 0
            while i < n-1:
                pair = (ele[i],ele[i+1])
                if pair not in self.bpe_encoding:
                    res.append(ele[i])
                    i += 1
                else:
                    res.append(self.bpe_encoding[pair])
                    i += 2
            if i == n-1:
                res.append(ele[i])            
            return res

        encoded = [updateEncoding(k) for k in encoded]
        return encoded
    
    def bytePairDecode(self, encoded):    
        
        def undoBpe(ele):
            i = 0
            n = len(ele)
            while i < n:
                if ele[i] not in self.bpe_decoding:
                    i += 1
                else:
                    insert = list(self.bpe_decoding[ele[i]])
                    ele = ele[:i] + insert + ele[i+1:]
                    n += 1
            return ele
            
        bpe_decoded = [undoBpe(k) for k in encoded]
        return bpe_decoded

class Dataset:
    def __init__(self, data):
        self.tokenizer = Tokenizer(data)
        self.data = data        
        self.encoded = self.tokenizer.encode(data)
        self.n = len(data)
    
    def split(self, group):
        n1 = int(self.n * 0.8)
        n2 = int(self.n * 0.9)

        if group == 'train':
            return self.encoded[:n1]        
        if group == 'test':
            return self.encoded[n1:n2]
        if group == 'validation':
            return self.encoded[n2:]        
        raise ValueError("invalid data split")        
    
    def summarizeEncoded(self):
        n = len([i for j in self.encoded for i in j])
        # print('tokens:', n)
        # print('vocab size:', len(self.tokenizer.encoding) + len(self.tokenizer.bpe_encoding))
        return n
    
    def bytePairEncode(self):
        return self.tokenizer.bytePairEncode(self.encoded)        
    
    def bytePairDecode(self):
        return self.tokenizer.bytePairDecode(self.encoded)        
    
d = Dataset(data)
orig = d.summarizeEncoded()

for _ in range(1):
    d.encoded = d.bytePairEncode()
    curr = d.summarizeEncoded()
    print('token reduction:', round(curr / orig,2))



token reduction: 0.98


In [7]:
# reconstruct back to original length
bpe_decoded = d.bytePairDecode()
bpe_decoded = len([i for j in bpe_decoded for i in j])

print('original len:', orig)
print('after bpe:', curr)
print('bpe decoded:', bpe_decoded)

for i,v in enumerate(d.encoded):
    if 97 in v: # 97 contains previously encoded value
        break

a = d.tokenizer.bytePairDecode([v])
print(d.tokenizer.decode(a)[0])
print(d.data[i])

original len: 3421277
after bpe: 3353680
bpe decoded: 3421277
Propeller Revolving shaft with blades, esp. For propelling a ship or aircraft.
Propeller Revolving shaft with blades, esp. For propelling a ship or aircraft.


In [8]:
print(d.tokenizer.decoding)
print(d.tokenizer.bpe_decoding)
d.tokenizer.encoding['<e>']

{0: '<s>', 1: '<e>', 2: ' ', 3: '!', 4: '"', 5: '$', 6: '%', 7: '&', 8: "'", 9: '(', 10: ')', 11: '+', 12: ',', 13: '-', 14: '.', 15: '/', 16: '0', 17: '1', 18: '2', 19: '3', 20: '4', 21: '5', 22: '6', 23: '7', 24: '8', 25: '9', 26: ':', 27: ';', 28: '=', 29: '?', 30: 'A', 31: 'B', 32: 'C', 33: 'D', 34: 'E', 35: 'F', 36: 'G', 37: 'H', 38: 'I', 39: 'J', 40: 'K', 41: 'L', 42: 'M', 43: 'N', 44: 'O', 45: 'P', 46: 'Q', 47: 'R', 48: 'S', 49: 'T', 50: 'U', 51: 'V', 52: 'W', 53: 'X', 54: 'Y', 55: 'Z', 56: '^', 57: 'a', 58: 'b', 59: 'c', 60: 'd', 61: 'e', 62: 'f', 63: 'g', 64: 'h', 65: 'i', 66: 'j', 67: 'k', 68: 'l', 69: 'm', 70: 'n', 71: 'o', 72: 'p', 73: 'q', 74: 'r', 75: 's', 76: 't', 77: 'u', 78: 'v', 79: 'w', 80: 'x', 81: 'y', 82: 'z', 83: '{', 84: '|', 85: '}'}
{86: (14, 2)}


1

In [220]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = torch.device('mps')
eos_tok = d.tokenizer.encoding['<e>']
torch.manual_seed(770)
batch_size = 64
block_size = 128
max_iters = 8000
eval_interval = 1000
eval_iters = 100
learning_rate = 3e-4
n_embd = 256
n_head = 4
n_layer = 3
dropout = 0.2


def transformData(split, dataset):    
    data = dataset.split(split)
    data = [i for j in data for i in j+[eos_tok]] # string separator for sequence packing
    return torch.tensor(data)

def getBatch(data):
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

@torch.no_grad()
def estimateLoss(dataset):
    out = {}
    model.eval()
    for split in ['train','validation']:
        losses = torch.zeros(eval_iters)
        data = transformData(split, dataset)
        for k in range(eval_iters):             
            X, Y = getBatch(data)
            logis, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


train = transformData('train', d)
xb,yb = getBatch(train)
d.tokenizer.decode([train[:200].tolist()])
print(xb,yb)

tensor([[70,  2, 76,  ..., 77, 76, 71],
        [76, 61, 74,  ..., 78, 61, 68],
        [66, 61, 59,  ..., 63, 57, 70],
        ...,
        [61, 70, 60,  ..., 65, 70, 76],
        [68, 68, 86,  ..., 70, 63,  2],
        [ 2, 59, 57,  ..., 75, 71, 70]], device='mps:0') tensor([[ 2, 76, 64,  ..., 76, 71, 70],
        [61, 74,  2,  ..., 61, 68, 71],
        [61, 59, 76,  ..., 57, 70, 65],
        ...,
        [70, 60, 75,  ..., 70, 76,  2],
        [68, 86, 31,  ..., 63,  2, 57],
        [59, 57, 77,  ..., 71, 70, 12]], device='mps:0')


In [221]:
torch.manual_seed(770)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask):        
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        # mask moved upstream so that it's only calculated once

        wei = q @ k.transpose(-2, -1) * C**-0.5   # (B, T, head_size) @ (B, head_size, T) --> (B, T, T)        
        wei = wei.masked_fill(mask, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = torch.cat([h(x, mask) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # projection layer back into residual pathway
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x_and_mask):
        x, mask = x_and_mask
        x = x + self.sa(self.ln1(x), mask)
        x = x + self.ffwd(self.ln2(x))
        return (x, mask)

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self.sa_head = MultiHeadAttention(4, n_embd//4)
        # self.ffwd = FeedForward(n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])            
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # masking with sequence packing via https://medium.com/@lukas.hauzenberger/efficient-llm-pretraining-packed-sequences-and-masked-attention-351d2a43b719        
        eos_idx = (idx.contiguous().view(-1) == eos_tok).nonzero(as_tuple=True)[0] + True
        arange = torch.arange(0,B*T+1,T).to(device)
        eos_idx_expanded = torch.cat([eos_idx, arange]).unique().sort()[0]
        normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
        normalized_idx = torch.where(normalized_idx == 0, T, normalized_idx)
        reps = normalized_idx[1:] - normalized_idx[:-1]
        reps = torch.where(reps < 1, normalized_idx[1:], reps)
        repeated_idx = torch.repeat_interleave(normalized_idx[1:], reps).view(B,1,T).expand(-1,T,-1)
        mask_indices = torch.arange(T).view(1,-1,1).expand(B, -1, T).to(device)
        mask = torch.ones(T, T, dtype=torch.bool).tril().expand(B, -1, -1).to(device)
        mask = ~mask.masked_fill(mask_indices >= repeated_idx, False)

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device))
        x = tok_emb + pos_emb
        # x = self.sa_head(x, idx)
        # x = self.ffwd(x)
        x, mask = self.blocks((x, mask))
        x = self.ln_f(x)        
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            # idx and targets are both nxm, where batch = n and time = m
            # vocab_size gives channel dimension
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

vocab_size = len(d.tokenizer.encoding) + len(d.tokenizer.bpe_encoding)
model = GPTLanguageModel(vocab_size)
m = model.to(device)

logits, loss = m(xb,yb)
print(logits.shape)
print(loss)

context = torch.zeros((5,1), dtype=torch.long, device = device)
g = m.generate(idx = context, max_new_tokens=100).tolist()
d.tokenizer.decode(g)

torch.Size([8192, 87])
tensor(4.5742, device='mps:0', grad_fn=<NllLossBackward0>)


["<s>b&)5K/orV?QVkXYxOmAZ&?p.FApO:8L%n6C{T&}. q+)eMT8vdTeA!m49yUZ'wt<e>Kfonsjv7tYg+&qM64d-z0Y)OtYZ6AVLW;FRME",
 "<s>n;i0sDlAC(d 84. f58,3I;66q/Ac=rK. s+v)pps1wgbGAEq-x|vg=eu<s>YQo9gg=1k2L1ktbZ2z:Ad0jXyu:xR9X<e>VL'1n(W)Igl9",
 '<s>/Or/m<e>Q%3Urpt29RRtA<e>TkPnp%8sqwn}QoafWu3(e?0lVY7;hYce8.w). :Yx9H-l:DU}ABO/13b. r)&UkS}wIqaynjU(9%8w/SSS',
 '<s>kXLIj. ,x3UN$/FB9y%DQn1%M|DVWfJ3i}8:+1\'kZ-Xnb<s>. n%gI+Zk:73NfMi24k5hwY\'AMj. \'F}{M<e>ebopre1"XV?^=1T-u1uZU$',
 '<s>H6Y=vS)3y9QQG7ha=+1F$&U0od67$1YiQ<e>3wrraTO29FF5="8:3w&.. LF:MllqW8\'S5fqwFOyy8d++)mNBNT;Y;BRMxdtW!HK-iX']

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr = learning_rate)
for step in range(max_iters+1):
    if step % eval_interval == 0:
        losses = estimateLoss(d)
        print(f"step {step}: trains loss {losses['train']:.4f}, val loss {losses['validation']:.4f}")

    xb,yb = getBatch(train)
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# step 0: trains loss 4.5111, val loss 4.5128
# step 2000: trains loss 2.6911, val loss 2.6618
# step 4000: trains loss 2.6040, val loss 2.5888
# step 6000: trains loss 2.5274, val loss 2.4745
# step 8000: trains loss 2.4696, val loss 2.5000
# step 10000: trains loss 2.4655, val loss 2.4518

# step 0: trains loss 4.5725, val loss 4.5723
# step 2000: trains loss 1.7361, val loss 1.7427
# step 4000: trains loss 1.5183, val loss 1.5325
# step 6000: trains loss 1.4301, val loss 1.4534
# step 8000: trains loss 1.3831, val loss 1.4136

step 0: trains loss 4.5725, val loss 4.5723
step 1000: trains loss 1.9623, val loss 1.9712
step 2000: trains loss 1.7361, val loss 1.7427
step 3000: trains loss 1.5988, val loss 1.6158
step 4000: trains loss 1.5183, val loss 1.5325
step 5000: trains loss 1.4584, val loss 1.4831
step 6000: trains loss 1.4301, val loss 1.4534
step 7000: trains loss 1.4082, val loss 1.4318
step 8000: trains loss 1.3831, val loss 1.4136


In [225]:
context = torch.zeros((10,1), dtype=torch.long, device = device)
g = m.generate(idx = context, max_new_tokens=100).tolist()
d.tokenizer.decode(g)

['<s>istacrial housewoman. 3 us Circraft nervous nervouss features blow wood at adon the recorded for a bo',
 '<s>Foll. By that goingly. To spoken with a joke-etc.<e>Unwated Not unwardined.<e>Deppense Closed defeatment o',
 '<s>at the goup saves, clarge, esp. 2 discontole of an arived in, tirricable.<e>Legseon 1 company scourage;',
 '<s> coined of groups etc. Aboughese.<e>Coloucle Exceeding inla0e.<e>Pit-bottone Pin-bodier-ped used of pitag',
 '<s>ised point.<e>Lacong size etc. To a basis or invertises. lake margo gradual.<e>Swide prestolic = 2 fold in',
 '<s> music inserity or innaviviges and received; insidence, counterports and tarandids. 2 receive, usu. Un',
 '<s> a by time. 2 a surplant gland. B persuit foot. 1 a noin, t bidding intensive joized and business. B suc',
 '<s> a hunted.<e>Thoughs Colloq. A shele.<e>Cluetical Each Fring evergrom offenses.<e>Ssound Esp. A pregnant of ',
 '<s>cloid incommodate you a this.<e>Dharenestly token or hypositive furnitude by ut ovalufully exami

In [ ]:
B,T,C = 2,3,4
x = torch.randn(B,T,C)
print(x) # see the dimensions
key = nn.Linear(C, 2, bias = False)
print(key)
k = key(x)
# key is 4x2
# x is 2x3x4
# 2x3x4 by 4x2 -> 2x3x2
# result of xA_transpose
print(k)

tensor([[[ 1.7426, -0.0910, -0.7546,  3.1379],
         [ 1.5918,  0.3563, -0.8177,  1.3310],
         [ 0.4366, -0.8934,  0.3827, -0.0597]],

        [[ 1.5830,  1.3877,  1.1179,  0.6330],
         [ 0.2439, -0.3213, -1.9929, -0.2173],
         [-0.4300, -0.9760,  0.4027,  0.4941]]])
Linear(in_features=4, out_features=2, bias=False)
tensor([[[-0.2824,  0.1743],
         [-0.2577,  0.0219],
         [-0.4122,  0.2443]],

        [[ 0.1529, -0.1452],
         [-0.3226,  0.0013],
         [-0.1414,  0.2176]]], grad_fn=<UnsafeViewBackward0>)


In [ ]:
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
k = key(x)      # (B, T, C) @ (C, head_size) --> (B, T, head_size)
q = query(x)    # (B, T, C) @ (C, head_size) --> (B, T, head_size)
print(k.shape)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, head_size) @ (B, head_size, T) --> (B, T, T)
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
wei[0]

torch.Size([4, 8, 16])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0476, 0.9524, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3886, 0.1774, 0.4341, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0681, 0.1738, 0.0329, 0.7251, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0145, 0.0997, 0.0154, 0.1153, 0.7552, 0.0000, 0.0000, 0.0000],
        [0.4947, 0.3203, 0.0556, 0.0556, 0.0595, 0.0144, 0.0000, 0.0000],
        [0.0382, 0.0481, 0.0421, 0.1695, 0.1880, 0.0474, 0.4666, 0.0000],
        [0.0284, 0.7178, 0.0204, 0.0278, 0.1596, 0.0024, 0.0022, 0.0414]],
       grad_fn=<SelectBackward0>)

In [141]:
# x = torch.tensor([[18,2,3,4,5,1,7,8], [18,2,4,4,5,3,7,8], [18,1,4,4,1,3,7,1]])
x = torch.tensor([[18,2,3,4,5,1,7,8], [18,2,4,4,5,3,7,8]])
# x = torch.tensor([[18,2,3,4,5,3,7,8]])
print((x.view(-1)==2).nonzero(as_tuple=True)[0])

B, T = x.shape
eos_idx = (x.view(-1) == d.tokenizer.encoding['<e>']).nonzero(as_tuple=True)[0] + True
eos_idx_expanded = torch.cat([eos_idx, torch.arange(0,B*T+1,T)]).unique().sort()[0]
normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
normalized_idx = torch.where(normalized_idx == 0, T, normalized_idx)
reps = normalized_idx[1:] - normalized_idx[:-1]
reps = torch.where(reps < 1, normalized_idx[1:], reps)
repeated_idx = torch.repeat_interleave(normalized_idx[1:], reps).view(B,1,T).expand(-1,T,-1)
mask_indices = torch.arange(T).view(1,-1,1).expand(B, -1, T)
mask = torch.ones(T, T, dtype=torch.bool).tril().expand(B, -1, -1)
mask = mask.masked_fill(mask_indices >= repeated_idx, False)
print(x)
print(mask)


tensor([1, 9])
tensor([[18,  2,  3,  4,  5,  1,  7,  8],
        [18,  2,  4,  4,  5,  3,  7,  8]])
tensor([[[ True, False, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False, False],
         [ True,  True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True,  True, False, False],
         [False, False, False, False, False, False,  True, False],
         [False, False, False, False, False, False,  True,  True]],

        [[ True, False, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False, False],
         [ True,  True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True,  True, False, False, False],
         [ True,  True,  Tr